# Faster R-CNN: two-stage parameter experiments + interpretability (Kaggle, ~9.5 h GPU)

All runs train on **every 5th train frame** with one shared schedule (`EPOCHS`, `BATCH`, input size) and are
evaluated on every 5th val frame. Each run changes **one** parameter from the baseline
(`--rpn-proposals 2000`, `--roi-pos-fraction 0.25`, `--box-loss smooth_l1`):

| Run | Change | Parameter | Cost |
|---|---|---|---|
| `base` | none | baseline | 1 training run (~2.8 h) |
| `giou` | `--box-loss giou` | box regression loss (stage 2) | 1 training run |
| `pos05` | `--roi-pos-fraction 0.5` | RoI sampling ratio (stage 2) | 1 training run, only if time is left |
| `base` @ 300 / 1000 proposals | test-time RPN proposals | number of RPN proposals (stage 1) | eval only |

Then `src.frcnn_explain` looks inside the `base` model: where each GT object is lost (RPN vs RoI head),
a t-SNE of RoI features, and Grad-CAM over ResNet stages and FPN levels.

Everything is logged to **Weights & Biases** (one W&B run per model: config, loss curves per loss term,
val metrics, val frames with GT and predicted boxes, explain plots).

Before running: **Settings -> Accelerator -> GPU**, **Internet on**, add the competition data
(**Add Input -> multiple-object-video-detection-hw-2-ucu-ai-2026**), and add your W&B API key as a
Kaggle secret named `WANDB_API_KEY` (**Add-ons -> Secrets**).
Use **Save Version -> Save & Run All** so `/kaggle/working/runs` survives the session (12 h limit).
Each run is evaluated right after training, so a timeout loses at most the run in progress.

In [1]:
!nvidia-smi
!git clone -q https://github.com/LiliAlania/drone-multi-object-detection.git /kaggle/working/repo
%cd /kaggle/working/repo
!pip install -q pycocotools wandb scikit-learn

Sat Nov 14 16:55:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os, sys, time
from pathlib import Path

T0 = time.time()
REPO = Path("/kaggle/working/repo")
sys.path.insert(0, str(REPO))
MOD = "src.models.faster_rcnn"
assert (REPO / "src/models/faster_rcnn.py").is_file(), "push the src/models/ layout to GitHub first"

EXPERIMENTS = {
    "base": "",
    "giou": "--box-loss giou",
    "pos05": "--roi-pos-fraction 0.5",  # runs only if the time budget allows
}
RUN = ["base", "giou", "pos05"]
PROPOSALS = [300, 1000]  # test-time sweep on `base`; its own eval is the 2000 point

EPOCHS = 5         # step LR drops after epochs 3 and 4; lower it if one epoch takes > 35 min
SUBSET_STEP = 5    # every 5th train frame: ~4.8k images, one epoch ~32 min on a T4
BATCH = 4          # 2 if CUDA out of memory
EVAL_STEP = 5      # every 5th val frame, as in the README
BUDGET_MIN = 690   # whole session; Kaggle stops at 12 h
RESERVE_MIN = 60   # kept free for the proposals sweep, the explain cells and the summary
RUNS = Path("/kaggle/working/runs")

# W&B: key from the Kaggle secret; the `!python` subprocesses inherit it through the environment
WANDB_ENTITY = "irynkasv-ukrainian-catholic-university-org"  # ...
if WANDB_ENTITY:
    os.environ["WANDB_ENTITY"] = WANDB_ENTITY

WANDB_PROJECT = "VisDrone_obj_det"
os.environ["WANDB_DIR"] = "/kaggle/working"  # offline runs then land in the notebook output
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
except Exception:  # secret not attached: log offline instead of crashing; upload later with `wandb sync`
    os.environ["WANDB_MODE"] = "offline"
    print("No WANDB_API_KEY secret: W&B logs offline into /kaggle/working/wandb")

In [3]:
# find the competition split folders without walking every image
INPUT = Path("/kaggle/input")
found = [p for depth in range(1, 5) for p in INPUT.glob("*/" * depth + "val/annotations_coco.json")]
assert found, "Competition data not found: add it with Add Input"
VAL_ROOT = found[0].parent
DATA = VAL_ROOT.parent
print("data:", DATA, sorted(p.name for p in DATA.iterdir()))

SUBSET = Path("/kaggle/working/visdrone-subset")
if not (SUBSET / "images" / "train").is_dir():
    !python -m src.make_subset --data-root {DATA} --output-dir {SUBSET} --step {SUBSET_STEP} --splits train val
    !rm -f {SUBSET}.zip
print(f"{len(list((SUBSET / 'images' / 'train').rglob('*.jpg')))} train images, "
      f"{(time.time() - T0) / 60:.0f} min elapsed")

data: /kaggle/input/competitions/multiple-object-video-detection-hw-2-ucu-ai-2026 ['test', 'test_images.csv', 'train', 'val']
Created 5439 frames; archive: /kaggle/working/visdrone-subset.zip
4868 train images, 2 min elapsed


In [4]:
run_minutes = []  # wall time of finished runs, to decide whether the next one still fits
done = []
for name in RUN:
    elapsed = (time.time() - T0) / 60
    if run_minutes and elapsed + max(run_minutes) > BUDGET_MIN - RESERVE_MIN:
        print(f"skip {name}: {elapsed:.0f} min used, a run takes ~{max(run_minutes):.0f} min")
        continue
    flags = EXPERIMENTS[name]
    out = RUNS / f"frcnn_{name}"
    resume = "--resume" if (out / "last.pt").is_file() else ""  # rerunning this cell continues an interrupted run
    print(f"=== {name}: {flags or 'baseline'} ===")
    start = time.time()
    !python -m {MOD} train --data-root {SUBSET} --epochs {EPOCHS} --batch {BATCH} --workers 4 {flags} --out-dir {out} {resume} --wandb-project {WANDB_PROJECT}
    !python -m {MOD} evaluate --weights {out}/last.pt --split-root {VAL_ROOT} --batch {BATCH} --step {EVAL_STEP} --wandb-project {WANDB_PROJECT} | tee {out}/eval.txt
    run_minutes.append((time.time() - start) / 60)
    done.append(name)
    print(f"{name}: {run_minutes[-1]:.0f} min, {(time.time() - T0) / 60:.0f} min elapsed")

=== base: baseline ===
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: irynkasv to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
W&B online init failed (you may not log runs directly to your organization, please try using your team entity); logging offline, upload later with `wandb sync`
wandb: WARNING `resume` will be ignored since W&B syncing is set to `offline`. Starting a new run with run id 64r1tvus.
wandb: Tracking run with wandb version 0.28.1
wandb: W&B syncing is set to `offline` in this directory. Run `wandb online` or set WANDB_MODE=online to enable cloud syncing.
wandb: Run data is saved locally in /kaggle/working/wandb/offline-run-20261114_165809-64r1tvus
wandb: View this run in the terminal with `wandb leet`
Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth" to /root/.cache/torch/hub/checkpoints/faster

In [5]:
# RPN proposals at test time: same `base` weights, only the number of proposals passed to the RoI head changes
import importlib
import torch

frcnn = importlib.import_module(MOD)

def with_proposals(n):
    # the checkpoint's saved model_cfg holds rpn_proposals=2000; override it after it is passed in
    def build(*args, **kwargs):
        return frcnn.build_model(*args, **{**kwargs, "rpn_proposals": n})
    return build

sweep = []
for n in PROPOSALS:
    print(f"=== base @ {n} proposals ===")
    metrics = frcnn.evaluate(RUNS / "frcnn_base" / "last.pt", VAL_ROOT, BATCH, EVAL_STEP, build_fn=with_proposals(n))
    sweep.append({"run": "base", "change": f"{n} RPN proposals (test)", "rpn_proposals": n, **metrics})
    torch.cuda.empty_cache()

=== base @ 300 proposals ===
loading annotations into memory...
Done (t=0.46s)
creating index...
index created!
Loading and preparing results...
DONE (t=0.05s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=14.96s).
Accumulating evaluation results...
DONE (t=0.80s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.234
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=500 ] = 0.475
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=500 ] = 0.196
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=500 ] = 0.090
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=500 ] = 0.221
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=500 ] = 0.418
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.105
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.348
 Average Recall     (AR) @[ IoU=0.5

In [6]:
# summary: mAP, AP50, AP_small and inference speed per run
import re
import pandas as pd
import wandb

rows = []
for name in done:
    text = (RUNS / f"frcnn_{name}" / "eval.txt").read_text(errors="ignore")
    ap = re.search(r"mAP@\[\.5:\.95\] ([\d.]+) \| AP50 ([\d.]+) \| AP_small ([\d.]+)", text)
    ms = re.search(r"Inference: ([\d.]+) ms/image", text)
    if not ap:
        print(f"{name}: no metrics in eval.txt")
        continue
    rows.append({"run": name, "change": EXPERIMENTS[name] or "baseline", "rpn_proposals": 2000,
                 "mAP": float(ap[1]), "AP50": float(ap[2]), "AP_small": float(ap[3]),
                 "ms_per_image": float(ms[1]) if ms else None})

cols = ["run", "change", "rpn_proposals", "mAP", "AP50", "AP_small", "ms_per_image"]
summary = pd.DataFrame(rows + sweep)[cols]
summary.to_csv(RUNS / "frcnn_experiments.csv", index=False)
print(f"settings: epochs {EPOCHS}, subset step {SUBSET_STEP}, batch {BATCH}, eval step {EVAL_STEP}")

# one comparison table across all runs in the W&B project
with wandb.init(project=WANDB_PROJECT, name="frcnn_experiments_summary", job_type="summary",
                config={"epochs": EPOCHS, "subset_step": SUBSET_STEP, "batch": BATCH, "eval_step": EVAL_STEP}) as run:
    run.log({"frcnn_experiments": wandb.Table(dataframe=summary)})
summary.round(4)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.


settings: epochs 5, subset step 5, batch 4, eval step 5


wandb: Currently logged in as: irynkasv to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


CommError: you may not log runs directly to your organization, please try using your team entity

## What the base model learns and where it fails

`objects` (every 5th val frame): for each GT object, the stage that lost it (no RPN proposal / proposal but
score < 0.3 / wrong class), what the RoI head says when given the exact GT box, and a t-SNE of the 1024-d RoI
features coloured by class, size and outcome. `gradcam` (every 50th val frame, 2 TP + 2 FP per frame):
Grad-CAM of single detection scores on C2..C5 and P2..P5. Plots land in `runs/frcnn_base/explain/` and in the
`frcnn_base` W&B run.

In [ ]:
from IPython.display import Image, display

BASE = RUNS / "frcnn_base" / "last.pt"
!python -m src.frcnn_explain objects --weights {BASE} --split-root {VAL_ROOT} --step {EVAL_STEP} --wandb-project {WANDB_PROJECT}
!python -m src.frcnn_explain gradcam --weights {BASE} --split-root {VAL_ROOT} --step 50 --wandb-project {WANDB_PROJECT}
for name in ["outcomes", "head_confusion", "tsne", "gradcam_levels", "gradcam_focus", "gradcam_gallery"]:
    display(Image(filename=str(BASE.parent / "explain" / f"{name}.png")))
print(f"{(time.time() - T0) / 60:.0f} min elapsed")

In [ ]:
# keep only checkpoints, eval logs and the summary CSV in the saved output
!rm -rf {SUBSET} /kaggle/working/repo